
# Optimizer r04 — linearized route–time bus allocation (Gurobi)

This notebook builds a clean MILP implementation of the linearized OC Transpo formulation using the modular files in `src/`:

- `src.demand_linear.LinearDemandModel`
- `src.costs.CostModel`
- `src.benefits_linear.LinearBenefitModel`
- `src.constraints.*`

The intent is to keep the notebook focused on:
1. loading the new dataset,
2. constructing coefficient dictionaries,
3. instantiating the model objects from `src/`,
4. building a compact Gurobi model, and
5. returning tidy solution tables for analysis.


In [22]:

from pathlib import Path
import math
import sys

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


In [23]:

# Make sure project imports resolve when this notebook is run either
# from the repository root or from a copied/exported location.

project_root_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/mnt/data'),
]

for root in project_root_candidates:
    src_path = root / 'src'
    if src_path.exists() and str(root) not in sys.path:
        sys.path.insert(0, str(root))
        break

try:
    import gurobipy as gp
    from gurobipy import GRB, quicksum
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        'gurobipy is not installed in this environment. Run this notebook in your '
        'local MECH 5800 project environment where Gurobi is configured.'
    ) from e

from src.demand_linear import LinearDemandModel
from src.costs import CostModel
from src.benefits_linear import LinearBenefitModel
from src.constraints import (
    add_integer_bus_vars,
    add_budget_constraint,
    add_fleet_constraints,
)



## 1. Load the subset

The route–time dataset is the main model input. The separate time-block parameter CSV is kept available for sensitivity work, but the route-level subset already contains the parameters needed by the current `src/` classes.


In [24]:

def first_existing_path(candidates):
    path = next((Path(p) for p in candidates if Path(p).exists()), None)
    if path is None:
        raise FileNotFoundError(f'Could not find any of: {candidates}')
    return path

route_data_path = first_existing_path([
    # 'data/route_timeblock_subset_v01.csv',
    # 'route_timeblock_subset_v01.csv',
    # '/mnt/data/route_timeblock_subset_v01.csv',

    # 'data/route_timeblock_v01.csv', # full data but was still missing some lengths
    # 'data/route_timeblock_subset_v02.csv', # reasonable subset

    'data/route_timeblock_v02.csv', # full data with all lengths filled in
])

param_data_path = first_existing_path([
    'data/timeblock_parameters_v01.csv',
    'timeblock_parameters_v01.csv',
    '/mnt/data/timeblock_parameters_v01.csv',
])

raw_df = pd.read_csv(route_data_path)
param_df = pd.read_csv(param_data_path)

print(f'Using route-time data: {route_data_path}')
print(f'Using time-block parameters: {param_data_path}')
raw_df.head()


Using route-time data: data\route_timeblock_v02.csv
Using time-block parameters: data\timeblock_parameters_v01.csv


,Route (r),Time Block (t),Route type,"Old ridership (x_old,r,t)",Trip length km (L_r),"Average length hr (T_r,t)","Old frequency of buses (f_old,r,t)","Old number of buses (n_old,r,t) - continuous","Old number of buses (n_old,r,t) - discrete","Time block hr (H_block,t)",Driver horuly wage rate $/hr (W_driver),Price of fuel $/litre (P_fuel),Fuel consumption (FC),Maintenace cost $/km (P_maintenance),Passenger hourly wage rate $/hr (W_passenger),VTTS fraction of passenger wage (F_wage),"Emission intensity of a car kgCO2eq/KPT (I_GHG,car)",Average Length of passenger trip (L_avg_trip),Social cost of carbon $/tonne CO2eq (SCC)
0,5,Early AM,Frequent,23,9.700,0.390,0.743,0.290,1.000,3.500,34.040,1.300,0.198,0.400,35.060,0.700,0.295,1.940,275.000
1,5,AM Peak,Frequent,292,9.700,0.549,4.680,2.569,3.000,2.500,34.040,1.300,0.242,0.400,35.060,1.000,0.330,1.940,275.000
2,5,Midday,Frequent,566,9.700,0.570,4.650,2.651,3.000,6.000,34.040,1.300,0.220,0.400,35.060,0.700,0.300,1.940,275.000
3,5,PM Peak,Frequent,303,9.700,0.657,5.743,3.773,4.000,3.500,34.040,1.300,0.242,0.400,35.060,1.000,0.330,1.940,275.000
4,5,Evening,Frequent,162,9.700,0.517,2.622,1.356,2.000,4.500,34.040,1.300,0.220,0.400,35.060,0.700,0.300,1.940,275.000


In [25]:
# Standardize the route-time dataset to concise internal names.

import pandas as pd

print("Exact raw headers from CSV:")
for c in raw_df.columns:
    print(repr(c))

col_map = {
    'Route (r)': 'route',
    'Time Block (t)': 'time_block',
    'Route type': 'route_type',
    'Old ridership (x_old,r,t)': 'x_old',
    'Trip length km (L_r)': 'L_r',
    'Average length hr (T_r,t)': 'T_rt',
    'Old frequency of buses (f_old,r,t)': 'f_old',
    'Old number of buses (n_old,r,t) - continuous': 'n_old_cont',
    'Old number of buses (n_old,r,t) - discrete': 'n_old',
    'Time block hr (H_block,t)': 'H_block',
    'Driver horuly wage rate $/hr (W_driver)': 'W_driver',
    'Price of fuel $/litre (P_fuel)': 'P_fuel',
    'Fuel consumption (FC)': 'fuel_consumption',
    'Maintenace cost $/km (P_maintenance)': 'P_maintenance',
    'Passenger hourly wage rate $/hr (W_passenger)': 'W_passenger',
    'VTTS fraction of passenger wage (F_wage)': 'F_wage',
    'Emission intensity of a car kgCO2eq/KPT (I_GHG,car)': 'I_GHG_car',
    'Average Length of passenger trip (L_avg_trip)': 'L_avg_trip',
    'Social cost of carbon $/tonne CO2eq (SCC)': 'SCC',
}

df = raw_df.rename(columns=col_map).copy()

required_cols = [
    'route', 'time_block', 'route_type',
    'x_old', 'L_r', 'T_rt', 'f_old',
    'n_old_cont', 'n_old',
    'H_block', 'W_driver', 'P_fuel',
    'fuel_consumption', 'P_maintenance',
    'W_passenger', 'F_wage',
    'I_GHG_car', 'L_avg_trip', 'SCC'
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"These required columns were not found after renaming: {missing}")

df['route'] = df['route'].astype(str).str.strip()
df['time_block'] = df['time_block'].astype(str).str.strip()
df['route_type'] = df['route_type'].astype(str).str.strip()


# Remove cancelled routes from this model version.
# They do not have a valid operating baseline and create missing values
# in fields required by the cost/benefit equations.
cancelled_mask = df['route_type'].str.strip().str.lower().eq('cancelled')

if cancelled_mask.any():
    print("Dropping cancelled route-time rows:")
    display(df.loc[cancelled_mask, ['route', 'time_block', 'route_type', 'x_old']])

df = df.loc[~cancelled_mask].copy()

numeric_cols = [
    'x_old', 'L_r', 'T_rt', 'f_old',
    'n_old_cont', 'n_old',
    'H_block', 'W_driver', 'P_fuel',
    'fuel_consumption', 'P_maintenance',
    'W_passenger', 'F_wage',
    'I_GHG_car', 'L_avg_trip', 'SCC'
]

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['n_old'] = df['n_old'].round().astype(int)
df['key'] = list(zip(df['route'], df['time_block']))

df[['route', 'time_block', 'route_type', 'x_old', 'n_old', 'T_rt', 'L_r']].head()

Exact raw headers from CSV:
'Route (r)'
'Time Block (t)'
'Route type'
'Old ridership (x_old,r,t)'
'Trip length km (L_r)'
'Average length hr (T_r,t)'
'Old frequency of buses (f_old,r,t)'
'Old number of buses (n_old,r,t) - continuous'
'Old number of buses (n_old,r,t) - discrete'
'Time block hr (H_block,t)'
'Driver horuly wage rate $/hr (W_driver)'
'Price of fuel $/litre (P_fuel)'
'Fuel consumption (FC)'
'Maintenace cost $/km (P_maintenance)'
'Passenger hourly wage rate $/hr (W_passenger)'
'VTTS fraction of passenger wage (F_wage)'
'Emission intensity of a car kgCO2eq/KPT (I_GHG,car)'
'Average Length of passenger trip (L_avg_trip)'
'Social cost of carbon $/tonne CO2eq (SCC)'
Dropping cancelled route-time rows:


,route,time_block,route_type,x_old
54,16,Early AM,Cancelled,9
55,16,AM Peak,Cancelled,308
56,16,Midday,Cancelled,682
57,16,PM Peak,Cancelled,293
58,16,Evening,Cancelled,157
...,...,...,...,...
763,291,AM Peak,Cancelled,137
764,291,Midday,Cancelled,0
765,291,PM Peak,Cancelled,121
766,291,Evening,Cancelled,14


,route,time_block,route_type,x_old,n_old,T_rt,L_r
0,5,Early AM,Frequent,23,1,0.390,9.700
1,5,AM Peak,Frequent,292,3,0.549,9.700
2,5,Midday,Frequent,566,3,0.570,9.700
3,5,PM Peak,Frequent,303,4,0.657,9.700
4,5,Evening,Frequent,162,2,0.517,9.700



## 2. Tuneable Scenario assumptions

- how far each route–time service level may move from current service,
- how much total fleet is available in each time block, and
- the total incremental operating budget for the subset.

In [26]:

# --- service adjustment bounds ---
# Frequent routes are allowed to move a bit more than local routes.
max_change_by_route_type = {
    'Frequent': 2,
    'Local': 1,
    'Connexion': 1,
}

# --- block-level fleet expansion above the current subset allocation ---
extra_buses_by_block = {
    'Early AM': 2,
    'AM Peak': 3,
    'Midday': 3,
    'PM Peak': 3,
    'Evening': 2,
    'Night': 2,
}

# --- incremental budget for this subset model ---
# CostModel returns change in operating cost relative to current service,
# so this is an allowable *increase* in daily operating cost.
budget_total = 3_000.0

# --- linearization control ---
# Based on the corrected sign convention that ridership rises with additional service.
# If x ~ x_old * (n_new / n_old)^elasticity, then dx/dn at n_old is elasticity * x_old / n_old.
demand_elasticity = 0.5


## 3. Build coefficient dictionaries for the `src/` models

For this linearized notebook, the coefficient construction is kept explicit:

- **Demand slope**
  $$
  \alpha_{r,t} \approx \varepsilon \frac{x^{\text{old}}_{r,t}}{n^{\text{old}}_{r,t}}
  $$

- **Time benefit per added bus** from a linearization of average waiting time
  $$
  \beta^{\text{time}}_{r,t}
  \approx
  x^{\text{old}}_{r,t}\, W_{\text{pass}} \, F_{\text{wage}}
  \frac{0.5\, T_{r,t}}{\left(n^{\text{old}}_{r,t}\right)^2}
  $$

- **Emission benefit per added bus** from induced ridership shifted from car travel
  $$
  \beta^{\text{em}}_{r,t}
  \approx
  \alpha_{r,t}\, L^{\text{avg}}_{\text{trip}} \, I_{\text{GHG,car}} \, \frac{SCC}{1000}
  $$

If you later want coefficients taken directly from the report or from Jessica's latest calibration, this is the one cell to swap out.

In [27]:
R = sorted(df['route'].unique())
T = ['Early AM', 'AM Peak', 'Midday', 'PM Peak', 'Evening', 'Night']
RT = [(r, t) for r in R for t in T]

# Quick integrity check: every route-time pair in RT must exist in the cleaned dataset
missing = set(RT) - set(df['key'])
if missing:
    raise ValueError(f"Missing route-time combinations in dataset: {sorted(missing)}")

# Quick integrity check: every route type in the dataset must have a change cap
route_types_in_df = sorted(df['route_type'].dropna().unique())
missing_route_types = sorted(set(route_types_in_df) - set(max_change_by_route_type.keys()))
if missing_route_types:
    raise KeyError(f"Missing max-change caps for route types: {missing_route_types}")

# Dictionaries keyed by (route, time_block)
x_old = df.set_index('key')['x_old'].to_dict()
n_old = df.set_index('key')['n_old'].astype(int).to_dict()
T_rt = df.set_index('key')['T_rt'].to_dict()
route_type = df.set_index('key')['route_type'].to_dict()

# Hard check: positive route times are required for any row with positive baseline service
bad_T_keys = [k for k in RT if pd.isna(T_rt[k]) or T_rt[k] < 0]
if bad_T_keys:
    raise ValueError(f"Invalid T_rt values found for keys: {bad_T_keys}")

# Dictionaries keyed by route or by time block
L_r = df.groupby('route')['L_r'].first().to_dict()
H_block = df.groupby('time_block')['H_block'].first().to_dict()

# Scalars currently treated as constant for compatibility with src interfaces
W_driver = float(df['W_driver'].iloc[0])
P_maintenance = float(df['P_maintenance'].iloc[0])

# CostModel currently expects scalar fuel consumption, so use the dataset-average value
fuel_consumption = float(df['fuel_consumption'].mean())
P_fuel = float(df['P_fuel'].iloc[0])

alpha = {}
beta_time = {}
beta_emissions = {}
n_min = {}
n_max = {}

frozen_zero_keys = []

for _, row in df.iterrows():
    k = row['key']
    x0 = float(row['x_old'])
    T0 = float(row['T_rt'])
    n0 = int(row['n_old'])
    rt_type = row['route_type']

    # If baseline service is zero, keep this block in the model
    # but force the decision variable to remain zero.
    if n0 == 0:
        alpha[k] = 0.0
        beta_time[k] = 0.0
        beta_emissions[k] = 0.0
        n_min[k] = 0
        n_max[k] = 0
        frozen_zero_keys.append(k)
        continue

    # Positive-service rows use the usual linearized coefficients
    if T0 <= 0:
        raise ValueError(f"T_rt must be positive when n_old > 0, but got T_rt={T0} for key {k}")

    if rt_type not in max_change_by_route_type:
        raise KeyError(f"Route type {rt_type!r} is missing from max_change_by_route_type")

    alpha[k] = demand_elasticity * x0 / n0
    beta_time[k] = x0 * float(row['W_passenger']) * float(row['F_wage']) * (0.5 * T0 / (n0 ** 2))
    beta_emissions[k] = alpha[k] * float(row['L_avg_trip']) * float(row['I_GHG_car']) * float(row['SCC']) / 1000.0

    delta_cap = int(max_change_by_route_type[rt_type])
    n_min[k] = max(0, n0 - delta_cap)
    n_max[k] = n0 + delta_cap

# Final completeness checks: every RT key must exist in every optimization dictionary
for name, d in {
    'x_old': x_old,
    'n_old': n_old,
    'T_rt': T_rt,
    'route_type': route_type,
    'alpha': alpha,
    'beta_time': beta_time,
    'beta_emissions': beta_emissions,
    'n_min': n_min,
    'n_max': n_max,
}.items():
    missing_keys = [k for k in RT if k not in d]
    if missing_keys:
        raise KeyError(f"{name} is missing keys: {missing_keys}")

# Keys that are actually decision variables in this model
RT_opt = [k for k in RT if not (n_old[k] == 0 and n_min[k] == 0 and n_max[k] == 0)]

print(f"Total RT keys: {len(RT)}")
print(f"Optimized RT keys: {len(RT_opt)}")
print(f"Frozen-at-zero RT keys: {len(RT) - len(RT_opt)}")

fleet_cap = {
    t: int(df.loc[df['time_block'] == t, 'n_old'].sum() + extra_buses_by_block[t])
    for t in T
}

if frozen_zero_keys:
    print("Zero-baseline route-time blocks fixed at n_new = 0:")
    display(pd.DataFrame(frozen_zero_keys, columns=['route', 'time_block']))

coef_df = pd.DataFrame({
    'route': [k[0] for k in RT],
    'time_block': [k[1] for k in RT],
    'route_type': [route_type[k] for k in RT],
    'x_old': [x_old[k] for k in RT],
    'n_old': [n_old[k] for k in RT],
    'T_rt': [T_rt[k] for k in RT],
    'alpha': [alpha[k] for k in RT],
    'beta_time': [beta_time[k] for k in RT],
    'beta_emissions': [beta_emissions[k] for k in RT],
    'n_min': [n_min[k] for k in RT],
    'n_max': [n_max[k] for k in RT],
})

coef_df.head(12)

Total RT keys: 510
Optimized RT keys: 429
Frozen-at-zero RT keys: 81
Zero-baseline route-time blocks fixed at n_new = 0:


,route,time_block
0,15,Night
1,21,Early AM
2,21,Midday
3,21,Evening
4,21,Night
5,23,Early AM
6,23,Midday
7,23,Evening
8,23,Night
9,26,Evening


,route,time_block,route_type,x_old,n_old,T_rt,alpha,beta_time,beta_emissions,n_min,n_max
0,10,Early AM,Frequent,42,1,0.550,21.000,283.460,5.997,0,3
1,10,AM Peak,Frequent,879,5,0.743,87.900,457.952,28.079,3,7
2,10,Midday,Frequent,2275,4,0.755,284.375,"1,317.311",82.582,2,6
3,10,PM Peak,Frequent,1291,5,0.877,129.100,793.904,41.240,3,7
4,10,Evening,Frequent,443,2,0.838,110.750,"1,138.853",32.162,0,4
5,10,Night,Frequent,229,1,0.637,114.500,"1,790.008",32.697,0,3
6,11,Early AM,Frequent,112,2,0.720,28.000,247.383,10.494,0,4
7,11,AM Peak,Frequent,829,5,1.003,82.900,583.039,34.757,3,7
8,11,Midday,Frequent,1989,5,1.098,198.900,"1,071.956",75.811,3,7
9,11,PM Peak,Frequent,1217,7,1.227,86.929,534.221,36.446,5,9


## 4. Instantiate the modular `src/` model objects

In [28]:

demand_model = LinearDemandModel(
    alpha=alpha,
    n_old=n_old,
    x_old=x_old,
)

cost_model = CostModel(
    H_block=H_block,
    W_driver=W_driver,
    L_r=L_r,
    T_rt=T_rt,
    P_fuel=P_fuel,
    fuel_consumption=fuel_consumption,
    P_maintenance=P_maintenance,
    n_old=n_old,
)

benefit_model = LinearBenefitModel(
    beta_time=beta_time,
    beta_emissions=beta_emissions,
    n_old=n_old,
)


In [29]:
import math
import pandas as pd

def is_bad_number(x):
    try:
        x = float(x)
        return not math.isfinite(x)
    except Exception:
        return True

bad_rows = []

for (r, t) in RT_opt:
    row_issues = []

    vals = {
        'x_old': x_old[(r, t)],
        'n_old': n_old[(r, t)],
        'T_rt': T_rt[(r, t)],
        'L_r': L_r[r],
        'H_block': H_block[t],
        'alpha': alpha[(r, t)],
        'beta_time': beta_time[(r, t)],
        'beta_emissions': beta_emissions[(r, t)],
        'n_min': n_min[(r, t)],
        'n_max': n_max[(r, t)],
        'W_driver': W_driver,
        'P_fuel': P_fuel,
        'fuel_consumption': fuel_consumption,
        'P_maintenance': P_maintenance,
    }

    for name, val in vals.items():
        if is_bad_number(val):
            row_issues.append((name, val))

    if row_issues:
        bad_rows.append({
            'route': r,
            'time_block': t,
            'issues': row_issues
        })

print(f"Bad RT_opt rows found: {len(bad_rows)}")

if bad_rows:
    bad_df = pd.DataFrame(bad_rows)
    display(bad_df)
    raise ValueError("NaN or Inf found in optimization inputs. Inspect bad_df above.")

Bad RT_opt rows found: 0


## 5. Build the Gurobi model

**Objective**
$$
\min \sum_{r,t} \left(\Delta C_{r,t} - \Delta B_{r,t}\right)
$$

**Constraints used here**
- integer bus counts,
- per-block fleet caps,
- total incremental operating budget,
- variable bounds around current service.

In [30]:
m = gp.Model('oc_transpo_linearized_r04')
m.Params.OutputFlag = 1

n_new = add_integer_bus_vars(
    model=m,
    RT=RT_opt,
    n_min=n_min,
    n_max=n_max,
    name='n_new',
)

objective = quicksum(
    cost_model.total(r, t, n_new[r, t]) - benefit_model.benefit_total(r, t, n_new[r, t])
    for r, t in RT_opt
)
m.setObjective(objective, GRB.MINIMIZE)

Set parameter OutputFlag to value 1


In [31]:

# src.constraints.add_fleet_constraints currently expects a scalar fleet cap,
# but this project needs a time-block-specific fleet limit.
# Build those block constraints cleanly here.

# Build fleet constraints only over decision variables that actually exist in RT_opt.
fleet_constraints = {
    t: m.addConstr(
        quicksum(n_new[r, tb] for (r, tb) in RT_opt if tb == t) <= fleet_cap[t],
        name=f'fleet[{t}]'
    )
    for t in T
}

add_budget_constraint(
    model=m,
    n_new=n_new,
    RT=RT_opt,
    cost_model=cost_model,
    budget_total=budget_total,
)

m.optimize()

print('Status code:', m.Status)
if m.Status == GRB.OPTIMAL:
    print('Optimal objective:', m.ObjVal)
    print('MIP gap:', m.MIPGap)
    print('Runtime (s):', m.Runtime)


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1195G7 @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2796207 - for non-commercial use only - registered to jo___@cmail.carleton.ca
Optimize a model with 7 rows, 429 columns and 858 nonzeros
Model fingerprint: 0xb60c6317
Variable types: 0 continuous, 429 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e+00, 2e+04]
  Bounds range     [1e+00, 1e+01]
  RHS range        [6e+01, 2e+05]
Found heuristic solution: objective -80967.42959
Presolve removed 0 rows and 105 columns
Presolve time: 0.00s
Presolved: 7 rows, 324 columns, 648 nonzeros
Variable types: 0 continuous, 324 integer (0 binary)

Root relaxation: objective -2.597588e+05, 6 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objectiv

## 6. Collect route–time results

In [32]:

if m.Status != GRB.OPTIMAL:
    raise RuntimeError(f'Model did not solve to optimality. Status = {m.Status}')

rows = []

for r, t in RT:
    active = (r, t) in RT_opt

    if active:
        n_star = int(round(n_new[r, t].X))
        x_new = demand_model.ridership(r, t, n_star)
        delta_x = demand_model.delta_ridership(r, t, n_star)
        cost_delta = cost_model.total(r, t, n_star)
        benefit_time_delta = benefit_model.benefit_time(r, t, n_star)
        benefit_emissions_delta = benefit_model.benefit_emissions(r, t, n_star)
        benefit_total_delta = benefit_model.benefit_total(r, t, n_star)
        net_contrib = cost_delta - benefit_total_delta
    else:
        n_star = 0
        x_new = x_old[(r, t)]
        delta_x = 0.0
        cost_delta = 0.0
        benefit_time_delta = 0.0
        benefit_emissions_delta = 0.0
        benefit_total_delta = 0.0
        net_contrib = 0.0

    rows.append({
        'route': r,
        'time_block': t,
        'route_type': route_type[(r, t)],
        'active_in_model': active,
        'n_old': n_old[(r, t)],
        'n_new': n_star,
        'delta_n': n_star - n_old[(r, t)],
        'x_old': x_old[(r, t)],
        'x_new_linear': x_new,
        'delta_ridership': delta_x,
        'operating_cost_delta': cost_delta,
        'benefit_time_delta': benefit_time_delta,
        'benefit_emissions_delta': benefit_emissions_delta,
        'benefit_total_delta': benefit_total_delta,
        'net_objective_contribution': net_contrib,
        'at_lower_bound': n_star == n_min[(r, t)],
        'at_upper_bound': n_star == n_max[(r, t)],
    })

solution_df = pd.DataFrame(rows)
print('Optimal objective:', m.ObjVal)
solution_df


Optimal objective: -259758.8145555929


,route,time_block,route_type,active_in_model,n_old,n_new,delta_n,x_old,x_new_linear,delta_ridership,operating_cost_delta,benefit_time_delta,benefit_emissions_delta,benefit_total_delta,net_objective_contribution,at_lower_bound,at_upper_bound
0,10,Early AM,Frequent,True,1,3,2,42,84.000,42.000,391.944,566.920,11.994,578.914,-186.970,False,True
1,10,AM Peak,Frequent,True,5,3,-2,879,703.200,-175.800,-251.449,-915.903,-56.158,-972.061,720.612,True,False
2,10,Midday,Frequent,True,4,6,2,2275,"2,843.750",568.750,600.378,"2,634.622",165.165,"2,799.787","-2,199.409",False,True
3,10,PM Peak,Frequent,True,5,7,2,1291,"1,549.200",258.200,334.649,"1,587.807",82.479,"1,670.287","-1,335.638",False,True
4,10,Evening,Frequent,True,2,4,2,443,664.500,221.500,436.029,"2,277.706",64.324,"2,342.030","-1,906.001",False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,99,AM Peak,Local,True,2,3,1,822,"1,027.500",205.500,141.555,"2,024.557",69.002,"2,093.559","-1,952.004",False,True
506,99,Midday,Local,True,2,3,1,659,823.750,164.750,349.280,"1,061.365",50.290,"1,111.655",-762.375,False,True
507,99,PM Peak,Local,True,2,3,1,952,"1,190.000",238.000,193.543,"2,490.768",79.914,"2,570.682","-2,377.139",False,True
508,99,Evening,Local,True,1,2,1,242,363.000,121.000,282.387,"1,312.555",36.935,"1,349.490","-1,067.103",False,True


In [33]:

summary = {
    'objective_value': m.ObjVal,
    'budget_total': budget_total,
    'budget_used': solution_df['operating_cost_delta'].sum(),
    'budget_slack': budget_total - solution_df['operating_cost_delta'].sum(),
    'total_delta_ridership': solution_df['delta_ridership'].sum(),
    'total_cost_delta': solution_df['operating_cost_delta'].sum(),
    'total_time_benefit_delta': solution_df['benefit_time_delta'].sum(),
    'total_emissions_benefit_delta': solution_df['benefit_emissions_delta'].sum(),
    'total_benefit_delta': solution_df['benefit_total_delta'].sum(),
    'vars_at_lower_bound': int(solution_df['at_lower_bound'].sum()),
    'vars_at_upper_bound': int(solution_df['at_upper_bound'].sum()),
    'runtime_seconds': m.Runtime,
    'node_count': m.NodeCount,
    'mip_gap': m.MIPGap,
}

for t in T:
    used = int(solution_df.loc[solution_df['time_block'] == t, 'n_new'].sum())
    summary[f'fleet_used[{t}]'] = used
    summary[f'fleet_cap[{t}]'] = fleet_cap[t]
    summary[f'fleet_slack[{t}]'] = fleet_cap[t] - used

pd.Series(summary)


objective_value                 -259,758.815
budget_total                       3,000.000
budget_used                       -3,827.181
budget_slack                       6,827.181
total_delta_ridership             28,048.002
total_cost_delta                  -3,827.181
total_time_benefit_delta         247,918.607
total_emissions_benefit_delta      8,013.026
total_benefit_delta              255,931.634
vars_at_lower_bound                  318.000
vars_at_upper_bound                  268.000
runtime_seconds                        0.036
node_count                             1.000
mip_gap                                0.000
fleet_used[Early AM]                  91.000
fleet_cap[Early AM]                   91.000
fleet_slack[Early AM]                  0.000
fleet_used[AM Peak]                  234.000
fleet_cap[AM Peak]                   234.000
fleet_slack[AM Peak]                   0.000
fleet_used[Midday]                   172.000
fleet_cap[Midday]                    172.000
fleet_slac

In [34]:

# Compact route-time allocation table for report use
allocation_pivot = solution_df.pivot(index='route', columns='time_block', values='n_new')
allocation_pivot = allocation_pivot[['Early AM', 'AM Peak', 'Midday', 'PM Peak', 'Evening', 'Night']]
allocation_pivot


time_block,Early AM,AM Peak,Midday,PM Peak,Evening,Night
route,,,,,,
10,3,3,6,7,4,3
11,1,7,7,9,5,3
110,1,5,3,4,3,0
111,3,5,5,5,4,3
12,3,7,7,5,1,3
138,0,2,0,2,0,0
14,3,7,6,7,4,3
15,2,4,2,4,2,0
153,0,0,0,0,0,0


In [35]:

# A more diagnostic view, sorted by the strongest incentive to add service.
solution_df.sort_values(['net_objective_contribution', 'route', 'time_block']).reset_index(drop=True)


,route,time_block,route_type,active_in_model,n_old,n_new,delta_n,x_old,x_new_linear,delta_ridership,operating_cost_delta,benefit_time_delta,benefit_emissions_delta,benefit_total_delta,net_objective_contribution,at_lower_bound,at_upper_bound
0,15,Midday,Local,True,1,2,1,2632,"3,948.000","1,316.000",282.994,"17,892.689",230.168,"18,122.857","-17,839.863",False,True
1,88,Night,Frequent,True,1,3,2,675,"1,350.000",675.000,435.968,"10,999.724",216.847,"11,216.572","-10,780.603",False,True
2,7,Night,Frequent,True,1,3,2,550,"1,100.000",550.000,404.972,"9,772.624",156.166,"9,928.790","-9,523.818",False,True
3,15,Evening,Local,True,1,2,1,902,"1,353.000",451.000,209.307,"6,452.902",78.880,"6,531.782","-6,322.474",False,True
4,88,Midday,Frequent,True,4,6,2,4712,"5,890.000","1,178.000",632.989,"5,247.251",384.853,"5,632.104","-4,999.115",False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,75,Night,Frequent,True,3,1,-2,487,324.667,-162.333,-492.097,"-1,134.108",-90.078,"-1,224.186",732.089,True,False
506,75,AM Peak,Frequent,True,7,5,-2,1541,"1,320.857",-220.143,-308.533,-935.006,-136.649,"-1,071.655",763.123,True,False
507,63,Midday,Frequent,True,3,1,-2,524,349.333,-174.667,-694.920,"-1,400.312",-98.276,"-1,498.588",803.668,True,False
508,90,Midday,Frequent,True,4,2,-2,1363,"1,022.250",-340.750,-617.585,"-1,432.110",-97.829,"-1,529.939",912.355,True,False



notes / next cleanup items

- `src.constraints.add_fleet_constraints` currently assumes a **single scalar fleet cap**, but this project needs a **different cap for each time block**.  (handled directly in notebook for now)
- `src.costs.CostModel` currently takes a **scalar** `fuel_consumption`, while the dataset gives it by time block. The notebook uses the subset-average value to stay compatible with the present `src/` interface.
- Possibly inconsistent with reportm so maybe change system to:
   - allow time-block-specific fleet caps inside `src.constraints`,
   - allow time-block-specific fuel consumption inside `src.costs`, and
   - replace the coefficient-construction cell with the report-calibrated linearization exactly as finalized.
